# 🚀 Titanic Survival Prediction: Modeling

This notebook implements the preprocessing pipeline and trains a baseline XGBoost model to predict passenger survival.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import sys
import os

# Add src to path so we can import our preprocessing pipeline
sys.path.append(os.path.abspath('../'))
from src.preprocessing import PreprocessingPipeline

print("Imports complete.")

## 1. Data Loading

In [ ]:
from src.data_loader import load_train, load_test

train_df = load_train()
test_df = load_test()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

## 2. Preprocessing

In [ ]:
pipeline = PreprocessingPipeline()

# Fit and transform training data
train_preprocessed = pipeline.fit_transform(train_df)

# Transform test data
test_preprocessed = pipeline.transform(test_df)

print("Preprocessing complete.")
print(f"Preprocessed Train shape: {train_preprocessed.shape}")
print(f"Preprocessed Test shape: {test_preprocessed.shape}")

## 3. Validation Split

In [ ]:
# Separate target and features
y = train_preprocessed['Survived']
X = train_preprocessed.drop(columns=['Survived', 'PassengerId'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

## 4. Baseline XGBoost Model

In [ ]:
model = XGBClassifier(
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=1000,
    random_state=42,
    eval_metric='logloss'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50, 
    verbose=False
)

print(f"Best iteration: {model.best_iteration}")

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_pred))
print("AUC-ROC:", roc_auc_score(y_val, y_prob))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 6. Final Prediction and Submission

In [ ]:
# Retrain on full training set
X_full = train_preprocessed.drop(columns=['Survived', 'PassengerId'])
y_full = train_preprocessed['Survived']

final_model = XGBClassifier(
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=model.best_iteration,
    random_state=42,
    eval_metric='logloss'
)
final_model.fit(X_full, y_full)

# Predict for test set
X_test = test_preprocessed.drop(columns=['PassengerId'])
test_preds = final_model.predict(X_test)

# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_preds
})

os.makedirs('../outputs', exist_ok=True)
submission.to_csv('../outputs/submission.csv', index=False)
print("Submission file saved to outputs/submission.csv")